# 04 — Architecture Comparison: BiGRU · CNN-LSTM · Transformer · Attn-GRU

## Motivation

Notebooks 02 and 03 established **BiGRU** as the best single model (MAE 0.0364, R² 0.863 on GroupKFold-6) 
and **CNN-LSTM** as a strong contender on CALCE folds but weak on NASA transfer.  
This notebook pits the four best architectures head-to-head on the same data and CV protocol,  
adding two attention-based approaches:

- **Transformer**: global self-attention, no recurrence — may learn chemistry-agnostic voltage features  
- **Attn-GRU**: BiGRU with learned attention pooling instead of mean+max — keeps sequential inductive bias, learns *where* to read

All models use **cosine LR schedule** and **smoothed early stopping** (es_window=5) introduced in nb03.

## Models

| Model | Architecture | Params |
|---|---|---|
| **BiGRU** | BiGRU(47) → masked mean+max pool → Dropout → Linear | ~14.9k |
| **CNN-LSTM** | Conv1d(8,k=5) + GroupNorm → BiLSTM(40) → masked mean+max pool | ~16.3k |
| **Transformer** | Linear → SinPE → TransformerEncoder(d=32, 4h, 2L) → masked pool | ~17k |
| **Attn-GRU** | BiGRU(47) → attention pooling → Dropout → Linear | ~14.9k |

## Validation

- **GroupKFold-6** (headline): whole-cell folds, study-group-stratified  
- **Leave-One-Dataset-Out (LODO)**: NASA ↔ CALCE transfer

## Experiments

1. RF baseline — GroupKFold-6  
2. BiGRU — GroupKFold-6  
3. CNN-LSTM — GroupKFold-6  
4. Transformer — GroupKFold-6  
5. Attn-GRU — GroupKFold-6  
6. GroupKFold-6 comparison table  
7. LODO — all five models  
8. LODO comparison table  

## Setup

In [ ]:
import sys
from pathlib import Path
from importlib import reload

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    else:
        start_path = Path(start_path)
    for marker in ['.git', 'pyproject.toml', 'CLAUDE.md', '.claude']:
        for p in [start_path] + list(start_path.parents):
            if (p / marker).exists():
                return p
    raise RuntimeError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import src.voltage_grid as vg
import src.vg_extended as vgx
import src.vg_models
reload(src.vg_models)

torch.manual_seed(42)
np.random.seed(42)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

RESULTS    = ROOT / "results" / "extended_tier1"
RESULTS.mkdir(parents=True, exist_ok=True)
CACHE      = ROOT / "data" / "processed" / "vg_extended_tier1.npz"

EPOCHS     = 300
PATIENCE   = 50
BATCH_SIZE = 64
LR         = 1e-3
SCHEDULER  = "cosine"

print(f"Cache:   {CACHE}")
print(f"Results: {RESULTS}")

## Load extended dataset

In [ ]:
X, mask, y, groups, cidx, ds_groups = vgx.load_npz_ext(CACHE)

print(f"X:         {X.shape}  float32")
print(f"mask:      {mask.shape}  bool")
print(f"y:         {y.shape}  [{y.min():.3f}, {y.max():.3f}]")
print(f"groups:    {len(set(groups))} unique cells")
print(f"ds_groups: {dict(zip(*np.unique(ds_groups, return_counts=True)))}")

In [ ]:
study_group_map = {
    "B0005": "NASA_CTRL",  "B0006": "NASA_CTRL",  "B0007": "NASA_CTRL",  "B0018": "NASA_CTRL",
    "RW1":   "NASA_RW",    "RW9":   "NASA_RW",    "RW13":  "NASA_RW",    "RW14":  "NASA_RW",
    "RW15":  "NASA_RW",    "RW16":  "NASA_RW",    "RW17":  "NASA_RW",    "RW19":  "NASA_RW",
    "RW20":  "NASA_RW",
    "CS2_8":  "CALCE_CS2_T1", "CS2_21": "CALCE_CS2_T1",
    "CS2_33": "CALCE_CS2_T1", "CS2_34": "CALCE_CS2_T1",
    "CS2_35": "CALCE_CS2_T2", "CS2_36": "CALCE_CS2_T2",
    "CS2_37": "CALCE_CS2_T2", "CS2_38": "CALCE_CS2_T2",
    "CS2_3":  "CALCE_CS2_T3", "CS2_9":  "CALCE_CS2_T3",
    "CX2_16": "CALCE_CX2_T1", "CX2_31": "CALCE_CX2_T1",
    "CX2_33": "CALCE_CX2_T1", "CX2_35": "CALCE_CX2_T1",
    "CX2_34": "CALCE_CX2_T2", "CX2_36": "CALCE_CX2_T2",
    "CX2_37": "CALCE_CX2_T2", "CX2_38": "CALCE_CX2_T2",
    "CX2_8":  "CALCE_CX2_T3",
}

study_groups = np.array([study_group_map.get(bid, "unknown") for bid in groups])

print(f"Study groups: {sorted(set(study_groups))}")
for sg in sorted(set(study_groups)):
    print(f"  {sg:25s}: {(study_groups == sg).sum():5d} samples")

## RF baseline

In [ ]:
feat, feat_names = vgx.vg_scalar_features(X, mask)
print(f"feat: {feat.shape}  features: {feat_names}")

print("\nRF — GroupKFold-6")
res_rf_gkf = vgx.run_rf_grouped_cv(
    feat, y, groups, cidx,
    GroupKFold(n_splits=6),
    cv_groups=study_groups,
)
agg_rf_gkf = vg.aggregate(res_rf_gkf)
print(f"  MAE  {agg_rf_gkf['mae']:.4f} ± {agg_rf_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_rf_gkf['rmse']:.4f} ± {agg_rf_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_rf_gkf['r2']:.4f} ± {agg_rf_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_rf_gkf['skill']:.4f}")
print(f"  Spearman  {agg_rf_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_rf_gkf,
    "RF — GroupKFold-6 by study group (SOH)",
    save_path=RESULTS / "04_rf_gkf_per_fold.png",
)

## BiGRU — `VGGRUReg`

Best single model from nb03. Bidirectional GRU → masked mean+max pool → Dropout → Linear.  
`VGGRUReg(n_features=3, hidden=47, dropout=0.35)` — ~14.9k params.

In [ ]:
demo = src.vg_models.VGGRUReg(n_features=3, hidden=47, dropout=0.35).to(DEVICE)
print(f"VGGRUReg params: {sum(p.numel() for p in demo.parameters()):,}")
with torch.no_grad():
    print(f"Forward shape: {demo(torch.tensor(X[:4]).to(DEVICE), torch.tensor(mask[:4]).to(DEVICE)).shape}")
del demo

In [ ]:
make_gru = lambda: src.vg_models.VGGRUReg(n_features=3, hidden=47, dropout=0.35)

print("BiGRU — GroupKFold-6")
res_gru_gkf = vgx.run_grouped_cv(
    make_gru, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_gru_gkf = vg.aggregate(res_gru_gkf)
print(f"\n  MAE  {agg_gru_gkf['mae']:.4f} ± {agg_gru_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_gru_gkf['rmse']:.4f} ± {agg_gru_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_gru_gkf['r2']:.4f} ± {agg_gru_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_gru_gkf['skill']:.4f}")
print(f"  Spearman  {agg_gru_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_gru_gkf,
    "BiGRU — GroupKFold-6 by study group (SOH)",
    save_path=RESULTS / "04_gru_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_gru_gkf,
    "BiGRU — GroupKFold-6 train/val loss (L1)",
    save_path=RESULTS / "04_gru_gkf_loss_curves.png",
)

## CNN-LSTM — `VGCNNLSTM`

Strong on CALCE folds (nb02). Conv1d front-end → BiLSTM → masked mean+max pool.  
`VGCNNLSTM(n_features=3, cnn_ch=8, hidden=40, dropout=0.35)` — ~16.3k params.

In [ ]:
demo = src.vg_models.VGCNNLSTM(n_features=3, cnn_ch=8, hidden=40, dropout=0.35).to(DEVICE)
print(f"VGCNNLSTM params: {sum(p.numel() for p in demo.parameters()):,}")
with torch.no_grad():
    print(f"Forward shape: {demo(torch.tensor(X[:4]).to(DEVICE), torch.tensor(mask[:4]).to(DEVICE)).shape}")
del demo

In [ ]:
make_cnnlstm = lambda: src.vg_models.VGCNNLSTM(n_features=3, cnn_ch=8, hidden=40, dropout=0.35)

print("CNN-LSTM — GroupKFold-6")
res_cnnlstm_gkf = vgx.run_grouped_cv(
    make_cnnlstm, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_cnnlstm_gkf = vg.aggregate(res_cnnlstm_gkf)
print(f"\n  MAE  {agg_cnnlstm_gkf['mae']:.4f} ± {agg_cnnlstm_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_cnnlstm_gkf['rmse']:.4f} ± {agg_cnnlstm_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_cnnlstm_gkf['r2']:.4f} ± {agg_cnnlstm_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_cnnlstm_gkf['skill']:.4f}")
print(f"  Spearman  {agg_cnnlstm_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_cnnlstm_gkf,
    "CNN-LSTM — GroupKFold-6 by study group (SOH)",
    save_path=RESULTS / "04_cnnlstm_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_cnnlstm_gkf,
    "CNN-LSTM — GroupKFold-6 train/val loss (L1)",
    save_path=RESULTS / "04_cnnlstm_gkf_loss_curves.png",
)

## Transformer — `VGTransformerReg`

Global self-attention, no recurrence. Fixed sinusoidal PE over voltage-grid positions.  
`VGTransformerReg(n_features=3, d_model=32, nhead=4, dim_feedforward=64, num_layers=2, dropout=0.35)` — ~17k params.

In [ ]:
demo = src.vg_models.VGTransformerReg(
    n_features=3, d_model=32, nhead=4, dim_feedforward=64, num_layers=2, dropout=0.35
).to(DEVICE)
print(f"VGTransformerReg params: {sum(p.numel() for p in demo.parameters()):,}")
with torch.no_grad():
    print(f"Forward shape: {demo(torch.tensor(X[:4]).to(DEVICE), torch.tensor(mask[:4]).to(DEVICE)).shape}")
del demo

In [ ]:
make_transformer = lambda: src.vg_models.VGTransformerReg(
    n_features=3, d_model=32, nhead=4, dim_feedforward=64, num_layers=2, dropout=0.35
)

print("Transformer — GroupKFold-6")
res_transformer_gkf = vgx.run_grouped_cv(
    make_transformer, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_transformer_gkf = vg.aggregate(res_transformer_gkf)
print(f"\n  MAE  {agg_transformer_gkf['mae']:.4f} ± {agg_transformer_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_transformer_gkf['rmse']:.4f} ± {agg_transformer_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_transformer_gkf['r2']:.4f} ± {agg_transformer_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_transformer_gkf['skill']:.4f}")
print(f"  Spearman  {agg_transformer_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_transformer_gkf,
    "Transformer — GroupKFold-6 by study group (SOH)",
    save_path=RESULTS / "04_transformer_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_transformer_gkf,
    "Transformer — GroupKFold-6 train/val loss (L1)",
    save_path=RESULTS / "04_transformer_gkf_loss_curves.png",
)

## Attn-GRU — `VGAttnGRUReg`

BiGRU with single-head attention pooling. Same parameter count as BiGRU — attention layer  
replaces the mean+max head exactly (4H+1 params either way).  
`VGAttnGRUReg(n_features=3, hidden=47, dropout=0.35)` — ~14.9k params.

In [ ]:
demo = src.vg_models.VGAttnGRUReg(n_features=3, hidden=47, dropout=0.35).to(DEVICE)
print(f"VGAttnGRUReg params: {sum(p.numel() for p in demo.parameters()):,}")
with torch.no_grad():
    print(f"Forward shape: {demo(torch.tensor(X[:4]).to(DEVICE), torch.tensor(mask[:4]).to(DEVICE)).shape}")
del demo

In [ ]:
make_attngru = lambda: src.vg_models.VGAttnGRUReg(n_features=3, hidden=47, dropout=0.35)

print("Attn-GRU — GroupKFold-6")
res_attngru_gkf = vgx.run_grouped_cv(
    make_attngru, X, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6),
    cv_groups=study_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_attngru_gkf = vg.aggregate(res_attngru_gkf)
print(f"\n  MAE  {agg_attngru_gkf['mae']:.4f} ± {agg_attngru_gkf['mae_std']:.4f}")
print(f"  RMSE {agg_attngru_gkf['rmse']:.4f} ± {agg_attngru_gkf['rmse_std']:.4f}")
print(f"  R²   {agg_attngru_gkf['r2']:.4f} ± {agg_attngru_gkf['r2_std']:.4f}")
print(f"  Skill     {agg_attngru_gkf['skill']:.4f}")
print(f"  Spearman  {agg_attngru_gkf['spearman']:.4f}")

In [ ]:
vgx.per_fold_scatter_ext(
    res_attngru_gkf,
    "Attn-GRU — GroupKFold-6 by study group (SOH)",
    save_path=RESULTS / "04_attngru_gkf_per_fold.png",
)
vgx.per_fold_loss_curves_ext(
    res_attngru_gkf,
    "Attn-GRU — GroupKFold-6 train/val loss (L1)",
    save_path=RESULTS / "04_attngru_gkf_loss_curves.png",
)

## GroupKFold-6 comparison

In [ ]:
ARM_ORDER_GKF = [
    ("RF",          res_rf_gkf,          agg_rf_gkf),
    ("BiGRU",       res_gru_gkf,         agg_gru_gkf),
    ("CNN-LSTM",    res_cnnlstm_gkf,     agg_cnnlstm_gkf),
    ("Transformer", res_transformer_gkf, agg_transformer_gkf),
    ("Attn-GRU",    res_attngru_gkf,     agg_attngru_gkf),
]
LOWER_BETTER = {"MAE", "RMSE"}

agg_str, agg_num = {}, {}
for arm, _, agg in ARM_ORDER_GKF:
    agg_str[arm] = {
        "MAE":      f"{agg['mae']:.4f} \u00b1 {agg['mae_std']:.4f}",
        "RMSE":     f"{agg['rmse']:.4f} \u00b1 {agg['rmse_std']:.4f}",
        "R\u00b2":  f"{agg['r2']:.4f} \u00b1 {agg['r2_std']:.4f}",
        "Skill":    f"{agg['skill']:.4f}",
        "Spearman": f"{agg['spearman']:.4f}",
    }
    agg_num[arm] = {
        "MAE":      agg["mae"],
        "RMSE":     agg["rmse"],
        "R\u00b2":  agg["r2"],
        "Skill":    agg["skill"],
        "Spearman": agg["spearman"],
    }

df_str = pd.DataFrame(agg_str).T
df_num = pd.DataFrame(agg_num).T

def _bold_better(df_display, df_values, lower_better):
    styled = df_display.copy()
    for col in df_display.columns:
        best_idx = df_values[col].idxmin() if col in lower_better else df_values[col].idxmax()
        styled.loc[best_idx, col] = f"**{df_display.loc[best_idx, col]}**"
    return styled

print("=== GroupKFold-6 aggregate results ===")
display(_bold_better(df_str, df_num, LOWER_BETTER))

df_str.to_csv(RESULTS / "04_comparison_gkf_metrics.csv")
print(f"Saved: {RESULTS / '04_comparison_gkf_metrics.csv'}")

fold_rows = []
for arm, res, _ in ARM_ORDER_GKF:
    for fi, fold in enumerate(res):
        m = fold["metrics"]
        fold_rows.append({
            "Model":     arm,
            "Fold":      fi + 1,
            "HeldGroup": ",".join(fold.get("held_group") or []),
            "HeldBids":  ",".join(fold.get("held_bids") or []),
            "MAE":       m["mae"],
            "RMSE":      m["rmse"],
            "R\u00b2":   m["r2"],
            "Skill":     m["skill"],
            "Spearman":  m["spearman"],
            "best_ep":   fold.get("best_epoch"),
        })

df_folds = pd.DataFrame(fold_rows)
df_pivot = df_folds.pivot_table(
    index=["Fold", "HeldGroup", "HeldBids"],
    columns="Model",
    values=["MAE", "RMSE", "R\u00b2", "Skill", "Spearman", "best_ep"],
).round(4)

print("\n=== Per-fold breakdown ===")
display(df_pivot)
df_pivot.to_csv(RESULTS / "04_per_fold_gkf_metrics.csv")
print(f"Saved: {RESULTS / '04_per_fold_gkf_metrics.csv'}")

## Leave-One-Dataset-Out (NASA ↔ CALCE transfer)

In [ ]:
print("RF — LODO")
res_rf_lodo = vgx.run_rf_grouped_cv(
    feat, y, groups, cidx,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
)
agg_rf_lodo = vg.aggregate(res_rf_lodo)
print(f"  MAE {agg_rf_lodo['mae']:.4f}  R² {agg_rf_lodo['r2']:.4f}")

In [ ]:
print("BiGRU — LODO")
res_gru_lodo = vgx.run_grouped_cv(
    make_gru, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_gru_lodo = vg.aggregate(res_gru_lodo)
print(f"  MAE {agg_gru_lodo['mae']:.4f}  R² {agg_gru_lodo['r2']:.4f}")

In [ ]:
print("CNN-LSTM — LODO")
res_cnnlstm_lodo = vgx.run_grouped_cv(
    make_cnnlstm, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_cnnlstm_lodo = vg.aggregate(res_cnnlstm_lodo)
print(f"  MAE {agg_cnnlstm_lodo['mae']:.4f}  R² {agg_cnnlstm_lodo['r2']:.4f}")

In [ ]:
print("Transformer — LODO")
res_transformer_lodo = vgx.run_grouped_cv(
    make_transformer, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_transformer_lodo = vg.aggregate(res_transformer_lodo)
print(f"  MAE {agg_transformer_lodo['mae']:.4f}  R² {agg_transformer_lodo['r2']:.4f}")

In [ ]:
print("Attn-GRU — LODO")
res_attngru_lodo = vgx.run_grouped_cv(
    make_attngru, X, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(),
    cv_groups=ds_groups,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
)
agg_attngru_lodo = vg.aggregate(res_attngru_lodo)
print(f"  MAE {agg_attngru_lodo['mae']:.4f}  R² {agg_attngru_lodo['r2']:.4f}")

In [ ]:
ARM_ORDER_LODO = [
    ("RF",          res_rf_lodo,          agg_rf_lodo),
    ("BiGRU",       res_gru_lodo,         agg_gru_lodo),
    ("CNN-LSTM",    res_cnnlstm_lodo,     agg_cnnlstm_lodo),
    ("Transformer", res_transformer_lodo, agg_transformer_lodo),
    ("Attn-GRU",    res_attngru_lodo,     agg_attngru_lodo),
]

lodo_rows = []
for arm, res, _ in ARM_ORDER_LODO:
    for fold in res:
        m = fold["metrics"]
        held_ds = ",".join(fold.get("held_group") or [])
        lodo_rows.append({
            "Model":        arm,
            "Held dataset": held_ds,
            "MAE":          round(m["mae"],      4),
            "RMSE":         round(m["rmse"],     4),
            "R\u00b2":      round(m["r2"],       4),
            "Skill":        round(m["skill"],    4),
            "Spearman":     round(m["spearman"], 4),
            "n":            m["n"],
        })

df_lodo = pd.DataFrame(lodo_rows)
print("=== Leave-One-Dataset-Out results ===")
display(df_lodo.pivot_table(
    index="Held dataset", columns="Model",
    values=["MAE", "RMSE", "R\u00b2", "Skill"], sort=False,
).round(4))

df_lodo.to_csv(RESULTS / "04_comparison_lodo_metrics.csv", index=False)
print(f"Saved: {RESULTS / '04_comparison_lodo_metrics.csv'}")

for arm, res, _ in ARM_ORDER_LODO:
    vgx.per_fold_scatter_ext(
        res,
        f"{arm} — LODO per-fold scatter (SOH)",
        save_path=RESULTS / f"04_{arm.lower().replace('-', '').replace(' ', '_')}_lodo_per_fold.png",
    )

## Conclusion

Fill in after running:

- **GroupKFold-6:** which architecture wins on CALCE folds? on NASA folds? does attention help transfer?
- **Attn-GRU vs BiGRU:** same param count — does learned attention pooling beat mean+max? if yes, the model is finding interpretable voltage positions to attend to.
- **Transformer:** does global self-attention generalise better across chemistries (lower LODO gap)? Hypothesis: yes, because it has no recurrence to accumulate dataset-specific sequential patterns.
- **LODO transfer gap** (GroupKFold MAE − LODO MAE): a smaller gap = more chemistry-agnostic representation.
- **Key slide:** GroupKFold-6 + LODO comparison table, best model highlighted.